# Fine-tuning experimental do NCM para Python

Este notebook adapta o **Next-Concept Modeling (NCM)** oficial ao laboratorio existente.

Estratégia em duas etapas:
- Stage 1: adapta o VQ-VAE XSum a um corpus maior de Python, preservando whitespace e estrutura;
- Stage 2: faz fine-tuning do gerador sobre o novo VQ-VAE;
- geração iterativa em blocos nativos de 48 tokens, chegando a 144 tokens;
- amostragem de varios candidatos seguida de `ast.parse`.

Por padrao, todos os treinos ficam desativados. Para corrigir o experimento atual sem reiniciar, use os blocos `Recuperacao incremental`: `RUN_VQVAE_RECOVERY=True` e depois `RUN_GENERATOR_RECOVERY=True`. Os blocos originais de Stage 1 e Stage 2 permanecem como histórico.

In [17]:
from pathlib import Path
import ast
import gc
import os
import random
import sys
import warnings

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GB")

torch: 2.6.0+cu124
CUDA disponivel: True
GPU: NVIDIA GeForce RTX 3060 Ti
VRAM total: 7.66 GB


In [3]:
import importlib.util

required = ["omegaconf", "hydra", "datasets", "huggingface_hub", "transformers"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Dependencias ausentes:", missing)
    print("Reconstruir a imagem com: docker compose up -d --build")
else:
    print("Dependencias essenciais encontradas.")

Dependencias essenciais encontradas.


In [4]:
import subprocess

NCM_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"
NCM_DIR = Path("/workspace/models/ncm")

if not (NCM_DIR / ".git").exists():
    NCM_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "https://github.com/HazyResearch/ncm.git", str(NCM_DIR)
    ], check=True)

subprocess.run(["git", "-C", str(NCM_DIR), "checkout", NCM_COMMIT], check=True)
sys.path.insert(0, str(NCM_DIR))
print("NCM:", NCM_DIR)
print("commit:", NCM_COMMIT)

NCM: /workspace/models/ncm
commit: c70274273f489abc73832b2b98584a8483b6b203


HEAD is now at c702742 update bibtex


In [5]:
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path("/workspace/models/ncm-checkpoints/xsum")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("config.yaml", "generator.pt", "tokenizer.pt"):
    path = hf_hub_download(
        repo_id="hazyresearch/ncm-checkpoints",
        filename=f"xsum/{filename}",
        local_dir=CHECKPOINT_DIR.parent,
    )
    downloaded = Path(path)
    if downloaded.parent != CHECKPOINT_DIR:
        downloaded.replace(CHECKPOINT_DIR / filename)

print("Checkpoints:", CHECKPOINT_DIR)
print([p.name for p in CHECKPOINT_DIR.iterdir()])

Checkpoints: /workspace/models/ncm-checkpoints/xsum
['tokenizer.pt', 'config.yaml', 'generator.pt']


In [6]:
from omegaconf import OmegaConf

CONFIG_PATH = CHECKPOINT_DIR / "config.yaml"
GENERATOR_PATH = CHECKPOINT_DIR / "generator.pt"
TOKENIZER_PATH = CHECKPOINT_DIR / "tokenizer.pt"

CFG = OmegaConf.load(CONFIG_PATH)
CFG.tokenizer.vqvae.checkpoint_path = str(TOKENIZER_PATH)

from tokenization.vqvae_tokenizer import VQVAETokenizer
from models.next_concept import NextConceptTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Este experimento precisa de GPU NVIDIA no container.")

def require_free_vram(min_free_gib):
    torch.cuda.empty_cache()
    free_gib = torch.cuda.mem_get_info()[0] / 2**30
    print("VRAM livre:", round(free_gib, 2), "GiB")
    if free_gib < min_free_gib:
        raise RuntimeError(
            f"VRAM livre insuficiente ({free_gib:.2f} GiB; minimo {min_free_gib:.2f} GiB). "
            "Pare o Ollama/LLM local ou reinicie o kernel antes de executar esta etapa."
        )

require_free_vram(0.8)
tk = VQVAETokenizer.from_config(CFG.tokenizer)
tk.vqvae.to(device)
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)

print("VQ-VAE carregado. O gerador sera alocado somente no Stage 2.")
print("VQ-VAE parametros:", round(sum(p.numel() for p in tk.vqvae.parameters()) / 1e6, 2), "M")

VRAM livre: 6.72 GiB
VQVAE, Total parameter count: 60.63M
VQVAE, Non embedding parameter count: 41.30M
VQ-VAE carregado. O gerador sera alocado somente no Stage 2.
VQ-VAE parametros: 60.63 M


## Corpus Python e preservacao estrutural

O corpus combina MBPP com um recorte de CodeSearchNet Python. O texto do codigo nao e normalizado: indentacao, quebras de linha, espacos e pontuacao sao preservados. A saida do checkpoint continua organizada em blocos nativos de 48 tokens.

In [7]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "sanitized")
PAD_ID = int(CFG.model.prefix_pad_token_id)
VQVAE_BLOCK_SIZE = int(tk.vqvae.context_length)
MAX_OUTPUT_BLOCKS = 3
CODESEARCH_TRAIN_ITEMS = 20_000

def normalize_item(prompt, code, source):
    prompt = str(prompt or "").strip()
    code = str(code or "")
    if not prompt or not code or not code.strip():
        return None
    prompt_ids = tk.pre_tokenizer.encode(prompt)
    code_ids = tk.pre_tokenizer.encode(code)
    if not prompt_ids or len(code_ids) < 1:
        return None
    if len(prompt_ids) > CFG.model.prefix_len:
        prompt = tk.pre_tokenizer.decode(prompt_ids[:CFG.model.prefix_len])
    return {"prompt": prompt, "code": code, "source": source}

corpus = [
    normalized
    for split in ("train", "validation", "test")
    for item in mbpp[split]
    if (normalized := normalize_item(item["prompt"], item["code"], "mbpp"))
]

code_search = load_dataset(
    "Nan-Do/code-search-net-python",
    split="train",
    streaming=True,
)
for item in code_search:
    prompt = item.get("summary") or item.get("docstring")
    code = item.get("code") or item.get("original_string")
    normalized = normalize_item(prompt, code, "codesearchnet")
    if normalized:
        corpus.append(normalized)
    if sum(example["source"] == "codesearchnet" for example in corpus) >= CODESEARCH_TRAIN_ITEMS:
        break

random.Random(SEED).shuffle(corpus)
train_end = int(len(corpus) * 0.94)
validation_end = train_end + int(len(corpus) * 0.03)
train_items = corpus[:train_end]
validation_items = corpus[train_end:validation_end]
test_items = corpus[validation_end:]
print("exemplos totais:", len(corpus))
print("treino:", len(train_items))
print("validacao:", len(validation_items))
print("teste:", len(test_items))
print("tamanhos por fonte:")
for source in sorted({item["source"] for item in corpus}):
    print(source, sum(item["source"] == source for item in corpus))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1100 > 1024). Running this sequence through the model will result in indexing errors


exemplos totais: 20420
treino: 19194
validacao: 612
teste: 614
tamanhos por fonte:
codesearchnet 20000
mbpp 420


## Diagnostico do VQ-VAE

Este passo mede o teto do tokenizador congelado antes de qualquer fine-tuning do gerador. `token_accuracy` e `syntax_valid_rate` precisam melhorar apos o Stage 1.

In [8]:
def split_code_blocks(code, max_blocks=MAX_OUTPUT_BLOCKS):
    ids = tk.pre_tokenizer.encode(code)
    return [
        ids[start:start + VQVAE_BLOCK_SIZE]
        for start in range(0, min(len(ids), VQVAE_BLOCK_SIZE * max_blocks), VQVAE_BLOCK_SIZE)
    ]

def padded_code_block(block):
    block = block[:VQVAE_BLOCK_SIZE]
    return block + [tk.pre_tokenizer.pad_token_id] * (VQVAE_BLOCK_SIZE - len(block))

@torch.no_grad()
def evaluate_vqvae(items, max_blocks=MAX_OUTPUT_BLOCKS):
    tk.vqvae.eval()
    exact = correct = total = syntax_valid = indentation_exact = 0
    samples = 0
    for item in items:
        reconstructed_parts = []
        for block in split_code_blocks(item["code"], max_blocks=max_blocks):
            padded = padded_code_block(block)
            source = torch.tensor(padded, device=device).unsqueeze(0)
            reconstruction = tk.vqvae.decode(tk.vqvae.encode(source)).argmax(dim=-1)
            original = source[:, :len(block)]
            decoded = reconstruction[:, :len(block)]
            correct += int((decoded == original).sum())
            total += original.numel()
            exact += int(torch.equal(decoded, original))
            reconstructed_parts.append(reconstruction.squeeze(0).tolist())
        reconstructed = tk.pre_tokenizer.decode(
            [token for part in reconstructed_parts for token in part],
            skip_special_tokens=True,
        ).strip()
        original_code = tk.pre_tokenizer.decode(
            [token for block in split_code_blocks(item["code"], max_blocks) for token in block],
            skip_special_tokens=True,
        ).strip()
        original_indents = [len(line) - len(line.lstrip(" \t")) for line in original_code.splitlines()]
        reconstructed_indents = [len(line) - len(line.lstrip(" \t")) for line in reconstructed.splitlines()]
        indentation_exact += int(original_indents == reconstructed_indents)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", SyntaxWarning)
                ast.parse(reconstructed)
            syntax_valid += 1
        except SyntaxError:
            pass
        samples += 1
    return {
        "token_accuracy": correct / max(total, 1),
        "exact_block_rate": exact / max(sum(len(split_code_blocks(item["code"], max_blocks)) for item in items), 1),
        "syntax_valid_rate": syntax_valid / max(samples, 1),
        "indentation_exact_rate": indentation_exact / max(samples, 1),
        "samples": samples,
    }

VQVAE_BASELINE = evaluate_vqvae(test_items[:100])
print("VQ-VAE baseline:", VQVAE_BASELINE)

VQ-VAE baseline: {'token_accuracy': 0.8821767713659605, 'exact_block_rate': 0.017006802721088437, 'syntax_valid_rate': 0.0, 'indentation_exact_rate': 0.13, 'samples': 100}


## Stage 1: adaptar o VQ-VAE ao Python

Treina encoder, decoder e quantizador do VQ-VAE por reconstrução token a token. Ative `RUN_VQVAE_TRAINING=True` somente para esta etapa; o gerador e o Stage 2 permanecem fora dela.

In [9]:
RUN_VQVAE_TRAINING = False
from torch.nn import functional as F
VQVAE_TRAIN_EPOCHS = 8
VQVAE_BATCH_SIZE = 2
VQVAE_ACCUMULATION_STEPS = 8
VQVAE_LEARNING_RATE = 1e-4
VQVAE_LOSS_WEIGHT = 1.0
VQVAE_MAX_TRAIN_ITEMS = 20_000
VQVAE_OUTPUT_DIR = Path("/workspace/results/ncm_codigo_full/vqvae")
VQVAE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vqvae_blocks = [
    padded_code_block(block)
    for item in train_items[:VQVAE_MAX_TRAIN_ITEMS]
    for block in split_code_blocks(item["code"])
]
random.Random(SEED).shuffle(vqvae_blocks)
STRUCTURAL_TOKEN_IDS = set()
for structure in ("\n", "\n    ", "\n\t", "    ", "\t", ":", "(", ")", "[", "]", "{", "}", ","):
    STRUCTURAL_TOKEN_IDS.update(tk.pre_tokenizer.encode(structure))
structural_tokens = torch.tensor(sorted(STRUCTURAL_TOKEN_IDS), device=device)
print("tokens estruturais ponderados:", sorted(STRUCTURAL_TOKEN_IDS))

if RUN_VQVAE_TRAINING:
    require_free_vram(3.5)
    tk.vqvae.train()
    for param in tk.vqvae.parameters():
        param.requires_grad_(True)
    optimizer = torch.optim.AdamW(tk.vqvae.parameters(), lr=VQVAE_LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    history = []

    for epoch in range(1, VQVAE_TRAIN_EPOCHS + 1):
        random.shuffle(vqvae_blocks)
        total_loss = 0.0
        steps = 0
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, len(vqvae_blocks), VQVAE_BATCH_SIZE):
            ids = torch.tensor(
                vqvae_blocks[start:start + VQVAE_BATCH_SIZE],
                dtype=torch.long,
                device=device,
            )
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits, vq_loss, _, utilization, _ = tk.vqvae(ids)
                reconstruction_token_loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    ids.reshape(-1),
                    ignore_index=tk.pre_tokenizer.pad_token_id,
                    reduction="none",
                )
                token_weights = torch.isin(ids.reshape(-1), structural_tokens).float() * 3.0 + 1.0
                active_tokens = ids.reshape(-1).ne(tk.pre_tokenizer.pad_token_id).float()
                reconstruction_loss = (
                    reconstruction_token_loss * token_weights * active_tokens
                ).sum() / (token_weights * active_tokens).sum().clamp_min(1.0)
                loss = reconstruction_loss + VQVAE_LOSS_WEIGHT * vq_loss
                loss = loss / VQVAE_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            steps += 1
            total_loss += loss.item() * VQVAE_ACCUMULATION_STEPS
            if steps % VQVAE_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(tk.vqvae.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            if steps % 500 == 0:
                print(f"stage1 epoca={epoch} passo={steps} loss={total_loss / steps:.4f} utilization={utilization:.4f}")

        if steps % VQVAE_ACCUMULATION_STEPS != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        metrics = evaluate_vqvae(test_items[:100])
        history.append({"epoch": epoch, "loss": total_loss / max(steps, 1), **metrics})
        torch.save({
            "model": tk.vqvae.state_dict(),
            "epoch": epoch,
            "history": history,
            "baseline": VQVAE_BASELINE,
        }, VQVAE_OUTPUT_DIR / f"tokenizer_code_epoch_{epoch}.pt")
        print("stage1 metrics:", metrics)

    CODE_VQVAE_CHECKPOINT = VQVAE_OUTPUT_DIR / f"tokenizer_code_epoch_{VQVAE_TRAIN_EPOCHS}.pt"
else:
    candidates = sorted(VQVAE_OUTPUT_DIR.glob("tokenizer_code_epoch_*.pt"))
    CODE_VQVAE_CHECKPOINT = candidates[-1] if candidates else None
    print("Stage 1 desativado. Checkpoint encontrado:", CODE_VQVAE_CHECKPOINT)

if CODE_VQVAE_CHECKPOINT is not None:
    code_vqvae_saved = torch.load(CODE_VQVAE_CHECKPOINT, map_location="cpu", weights_only=False)
    tk.vqvae.load_state_dict(code_vqvae_saved["model"], strict=True)
    print("VQ-VAE de codigo carregado:", CODE_VQVAE_CHECKPOINT)
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)
if 'optimizer' in globals():
    del optimizer
if 'scaler' in globals():
    del scaler
gc.collect()
torch.cuda.empty_cache()

tokens estruturais ponderados: [7, 8, 11, 25, 58, 60, 90, 92, 197, 198, 220]
Stage 1 desativado. Checkpoint encontrado: /workspace/results/ncm_codigo_full/vqvae/tokenizer_code_epoch_8.pt
VQ-VAE de codigo carregado: /workspace/results/ncm_codigo_full/vqvae/tokenizer_code_epoch_8.pt


In [10]:
require_free_vram(1.5)
net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
generator_base = torch.load(GENERATOR_PATH, map_location="cpu", weights_only=False)
net.load_state_dict(generator_base["model"], strict=True)
net.eval()

def expand_generator_examples(items):
    expanded = []
    for item in items:
        previous_code = ""
        for block_index, block in enumerate(split_code_blocks(item["code"])):
            prompt_text = item["prompt"]
            if previous_code:
                prompt_text += "\n\n# Continue the Python implementation:\n" + previous_code
            prompt_tokens = tk.pre_tokenizer.encode(prompt_text)
            task_tokens = tk.pre_tokenizer.encode(item["prompt"])
            available_previous = net.prefix_len - min(len(task_tokens), net.prefix_len)
            if len(prompt_tokens) > net.prefix_len and available_previous > 0:
                prompt_tokens = task_tokens[:net.prefix_len - available_previous] + prompt_tokens[-available_previous:]
            expanded.append({
                **item,
                "prompt_ids": prompt_tokens[:net.prefix_len],
                "code": tk.pre_tokenizer.decode(block, skip_special_tokens=True),
                "block_index": block_index,
            })
            previous_code += tk.pre_tokenizer.decode(block, skip_special_tokens=True)
    return expanded

generator_train_items = expand_generator_examples(train_items)
generator_validation_items = expand_generator_examples(validation_items)
generator_test_items = expand_generator_examples(test_items)
print("pares gerador:", len(generator_train_items), len(generator_validation_items), len(generator_test_items))

@torch.no_grad()
def make_training_batch(items):
    prefix_chunks = []
    ncm_chunks = []
    target_chunks = []
    prefix_mask_chunks = []

    for item in items:
        prompt_ids = list(item.get("prompt_ids", tk.pre_tokenizer.encode(item["prompt"])))
        prompt_ids = prompt_ids[:net.prefix_len]
        prompt_ids = prompt_ids + [PAD_ID] * (net.prefix_len - len(prompt_ids))
        prefix = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        prefix_mask_chunks.append(prefix.eq(PAD_ID))

        code_ids = tk.pre_tokenizer.encode(item["code"])
        code_ids = padded_code_block(code_ids)
        code_tensor = torch.tensor(code_ids, dtype=torch.long, device=device).unsqueeze(0)

        target = tk.vqvae.encode(code_tensor)
        ncm_input = tk.vqvae.vq.idx_to_ncm_input(target)
        if net.query_level0:
            l0_query = net.build_l0_query_input(1, dtype=ncm_input.dtype)
            ncm_input = torch.cat([l0_query, ncm_input], dim=1)
        prefix_embedding = tk.vqvae.encoder.wte(prefix)
        ncm_chunks.append(ncm_input)
        prefix_chunks.append(prefix_embedding)
        target_chunks.append(target)

    prefix_embedding = torch.cat(prefix_chunks, dim=0)
    ncm_input = torch.cat(ncm_chunks, dim=0)
    model_input = torch.cat([prefix_embedding, ncm_input], dim=1)
    targets = torch.cat(target_chunks, dim=0)
    prefix_pad_mask = torch.cat(prefix_mask_chunks, dim=0)
    return model_input, targets, prefix_pad_mask

smoke_items = generator_train_items[:1]
smoke_input, smoke_targets, smoke_prefix_pad_mask = make_training_batch(smoke_items)
net.eval()
with torch.inference_mode():
    smoke_logits = net.forward(smoke_input, prefix_pad_mask=smoke_prefix_pad_mask)

print("entrada:", tuple(smoke_input.shape))
print("alvos:", tuple(smoke_targets.shape))
print("logits:", tuple(smoke_logits.shape))
assert smoke_input.shape[1] == net.prefix_len + sum(net.levels)
assert smoke_logits.shape[:2] == smoke_targets.shape
print("Smoke test: OK")

VRAM livre: 6.39 GiB
NCP Transformer, Total parameter count: 88.02M
NCP Transformer, Non embedding parameter count: 87.71M
pares gerador: 55947 1779 1779
entrada: (1, 712, 384)
alvos: (1, 200)
logits: (1, 200, 4096)
Smoke test: OK


## Stage 2: fine-tuning do gerador NCM

A celula abaixo treina somente o gerador sobre o VQ-VAE já adaptado. Cada exemplo inclui o bloco de codigo alvo de 48 tokens e os blocos anteriores no prefixo. Checkpoints sao gravados em `/workspace/results/ncm_codigo_full/generator`.

In [12]:
import matplotlib.pyplot as plt
from torch.nn import functional as F

RUN_GENERATOR_TRAINING = False
BATCH_SIZE = 1
ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
TOTAL_EPOCHS = 20
MAX_TRAIN_ITEMS = len(generator_train_items)
PATIENCE = 12

OUTPUT_DIR = Path("/workspace/results/ncm_codigo_full/generator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_loss_history = []
validation_loss_history = []

if RUN_GENERATOR_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Treino interrompido: CUDA indisponivel.")

    net.train()
    net.vqvae_local[0].eval()
    optimizer = torch.optim.AdamW(
        (p for p in net.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    optimizer.zero_grad(set_to_none=True)

    checkpoints = list(OUTPUT_DIR.glob("epoch_*.pt"))
    if checkpoints:
        latest_path = max(checkpoints, key=lambda path: int(path.stem.split("_")[-1]))
        saved = torch.load(latest_path, map_location="cpu", weights_only=False)
        net.load_state_dict(saved["model"], strict=True)
        start_epoch = int(saved.get("epoch", 0))
        train_loss_history = list(saved.get("train_loss_history", []))
        validation_loss_history = list(saved.get("validation_loss_history", []))
        if saved.get("optimizer") is not None:
            optimizer.load_state_dict(saved["optimizer"])
        print("retomando de:", latest_path, "epoca:", start_epoch)
    else:
        start_epoch = 0

    selected = generator_train_items[:MAX_TRAIN_ITEMS]
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    best_val_loss = min(validation_loss_history) if validation_loss_history else float("inf")
    bad_epochs = 0

    for epoch in range(start_epoch + 1, TOTAL_EPOCHS + 1):
        random.shuffle(selected)
        epoch_loss = 0.0
        steps = 0

        for start in range(0, len(selected), BATCH_SIZE):
            batch_items = selected[start:start + BATCH_SIZE]
            model_input, targets, prefix_pad_mask = make_training_batch(batch_items)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                ) / ACCUMULATION_STEPS

            scaler.scale(loss).backward()
            steps += 1

            if steps % ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * ACCUMULATION_STEPS
            if steps % 8 == 0:
                print(f"epoca={epoch} passo={steps} loss={epoch_loss / steps:.4f}")

        if steps % ACCUMULATION_STEPS != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        mean_loss = epoch_loss / max(steps, 1)
        net.eval()
        validation_loss = 0.0
        validation_steps = 0
        with torch.inference_mode():
            for start in range(0, len(generator_validation_items), BATCH_SIZE):
                batch_items = generator_validation_items[start:start + BATCH_SIZE]
                model_input, targets, prefix_pad_mask = make_training_batch(batch_items)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                    loss = F.cross_entropy(
                        logits.float().reshape(-1, logits.shape[-1]),
                        targets.reshape(-1),
                    )
                validation_loss += loss.item()
                validation_steps += 1
        mean_validation_loss = validation_loss / max(validation_steps, 1)
        scheduler.step(mean_validation_loss)
        train_loss_history.append(mean_loss)
        validation_loss_history.append(mean_validation_loss)
        net.train()
        net.vqvae_local[0].eval()

        checkpoint_path = OUTPUT_DIR / f"epoch_{epoch}.pt"
        torch.save({
            "model": net.state_dict(),
            "epoch": epoch,
            "loss": mean_loss,
            "validation_loss": mean_validation_loss,
            "train_loss_history": train_loss_history,
            "validation_loss_history": validation_loss_history,
            "optimizer": optimizer.state_dict(),
            "config": OmegaConf.to_container(CFG, resolve=True),
        }, checkpoint_path)
        print(
            f"epoca={epoch} train_loss={mean_loss:.4f} "
            f"validation_loss={mean_validation_loss:.4f} checkpoint={checkpoint_path}"
        )

        if mean_validation_loss < best_val_loss:
            best_val_loss = mean_validation_loss
            bad_epochs = 0
            torch.save({
                "model": net.state_dict(),
                "epoch": epoch,
                "validation_loss": mean_validation_loss,
                "vqvae_checkpoint": str(CODE_VQVAE_CHECKPOINT),
                "config": OmegaConf.to_container(CFG, resolve=True),
            }, OUTPUT_DIR / "best.pt")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Parada antecipada sem melhoria por {PATIENCE} epocas.")
                break

    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, marker="o", label="treino")
    plt.plot(range(1, len(validation_loss_history) + 1), validation_loss_history, marker="o", label="validacao")
    plt.xlabel("Epoca")
    plt.ylabel("Cross entropy")
    plt.legend()
    plt.title("Perda de treino NCM - Python")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "loss_curve.png")
    plt.show()
else:
    print("Stage 2 desativado. Ajuste RUN_GENERATOR_TRAINING=True depois do Stage 1 e do smoke test.")

Stage 2 desativado. Ajuste RUN_GENERATOR_TRAINING=True depois do Stage 1 e do smoke test.


## Recuperacao incremental: completar o Stage 1

Esta etapa continua exclusivamente do ultimo VQ-VAE salvo. Ela aumenta o peso de delimitadores, novas linhas e indentacao para elevar a taxa de AST valida sem repetir as oito epocas anteriores. Ative `RUN_VQVAE_RECOVERY=True` somente depois de reiniciar o kernel com VRAM livre.

In [19]:
RUN_VQVAE_RECOVERY = True
VQVAE_RECOVERY_PLAIN_EPOCHS = 4
VQVAE_RECOVERY_STRUCTURAL_EPOCHS = 2
VQVAE_RECOVERY_EXTRA_EPOCHS = VQVAE_RECOVERY_PLAIN_EPOCHS + VQVAE_RECOVERY_STRUCTURAL_EPOCHS
VQVAE_RECOVERY_LR = 5e-6
VQVAE_RECOVERY_STRUCTURAL_LR = 2e-6
VQVAE_RECOVERY_BATCH_SIZE = 8
VQVAE_RECOVERY_ACCUMULATION = 2
VQVAE_RECOVERY_STRUCTURAL_MULTIPLIER = 2.0
RECOVERY_RUN_NAME = "frozen_quantizer_complete48"
RECOVERY_VQVAE_DIR = (
    Path("/workspace/results/ncm_codigo_full/recovery_vqvae")
    / RECOVERY_RUN_NAME
)
RECOVERY_VQVAE_DIR.mkdir(parents=True, exist_ok=True)

def complete_python_48_tokens(item):
    token_ids = tk.pre_tokenizer.encode(item["code"])
    if not 1 <= len(token_ids) <= VQVAE_BLOCK_SIZE:
        return False
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", SyntaxWarning)
            ast.parse(item["code"])
        return True
    except (SyntaxError, TypeError):
        return False

recovery_train_items = [item for item in train_items if complete_python_48_tokens(item)]
recovery_test_items = [item for item in test_items if complete_python_48_tokens(item)]
recovery_vqvae_blocks = [
    padded_code_block(tk.pre_tokenizer.encode(item["code"]))
    for item in recovery_train_items
]
random.Random(SEED).shuffle(recovery_vqvae_blocks)
print("exemplos completos ate 48 tokens:", {
    "train": len(recovery_train_items),
    "test": len(recovery_test_items),
})

source_vqvae = CODE_VQVAE_CHECKPOINT
if source_vqvae is None:
    raise FileNotFoundError("Treine ou carregue o Stage 1 antes da recuperacao.")

recovery_vqvae_checkpoints = sorted(
    RECOVERY_VQVAE_DIR.glob("vqvae_recovery_epoch_*.pt"),
    key=lambda path: int(path.stem.split("_")[-1]),
)
recovery_vqvae_source = recovery_vqvae_checkpoints[-1] if recovery_vqvae_checkpoints else source_vqvae
vqvae_recovery_saved = torch.load(recovery_vqvae_source, map_location="cpu", weights_only=False)
tk.vqvae.load_state_dict(vqvae_recovery_saved["model"], strict=True)
vqvae_recovery_start = int(vqvae_recovery_saved.get("epoch", 0))
print("VQ-VAE de recuperacao:", recovery_vqvae_source, "epoca:", vqvae_recovery_start)
tk.vqvae.eval()
source_recovery_metrics = evaluate_vqvae(recovery_test_items, max_blocks=1)
print("metricas de partida comparaveis:", source_recovery_metrics)

if RUN_VQVAE_RECOVERY:
    require_free_vram(3.5)
    tk.vqvae.train()
    tk.vqvae.vq.eval()
    for param in tk.vqvae.parameters():
        param.requires_grad_(True)
    for embedding in tk.vqvae.vq.embeddings:
        for param in embedding.parameters():
            param.requires_grad_(False)
    recovery_optimizer = torch.optim.AdamW(
        (param for param in tk.vqvae.parameters() if param.requires_grad),
        lr=VQVAE_RECOVERY_LR,
        weight_decay=0.01,
    )
    if vqvae_recovery_saved.get("optimizer") is not None:
        recovery_optimizer.load_state_dict(vqvae_recovery_saved["optimizer"])
    recovery_scaler = torch.amp.GradScaler("cuda", enabled=True)
    if vqvae_recovery_saved.get("scaler") is not None:
        recovery_scaler.load_state_dict(vqvae_recovery_saved["scaler"])
    recovery_history = list(vqvae_recovery_saved.get("recovery_history", []))
    best_reconstruction = max(
        recovery_history,
        key=lambda row: (
            row["syntax_valid_rate"],
            row["exact_block_rate"],
            row["token_accuracy"],
        ),
        default=source_recovery_metrics,
    )

    for recovery_epoch in range(
        vqvae_recovery_start + 1,
        vqvae_recovery_start + VQVAE_RECOVERY_EXTRA_EPOCHS + 1,
    ):
        random.shuffle(recovery_vqvae_blocks)
        structural_phase = recovery_epoch - vqvae_recovery_start > VQVAE_RECOVERY_PLAIN_EPOCHS
        phase_lr = VQVAE_RECOVERY_STRUCTURAL_LR if structural_phase else VQVAE_RECOVERY_LR
        for group in recovery_optimizer.param_groups:
            group["lr"] = phase_lr
        print("fase:", "estrutural" if structural_phase else "cross-entropy", "lr:", phase_lr)
        recovery_optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        steps = 0
        for start in range(0, len(recovery_vqvae_blocks), VQVAE_RECOVERY_BATCH_SIZE):
            ids = torch.tensor(
                recovery_vqvae_blocks[start:start + VQVAE_RECOVERY_BATCH_SIZE],
                dtype=torch.long,
                device=device,
            )
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits, vq_loss, _, utilization, _ = tk.vqvae(ids)
                token_loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    ids.reshape(-1),
                    reduction="none",
                )
                active = ids.reshape(-1).ne(tk.pre_tokenizer.pad_token_id).float()
                structural = torch.isin(ids.reshape(-1), structural_tokens).float()
                plain_reconstruction_loss = (token_loss * active).sum() / active.sum().clamp_min(1.0)
                if structural_phase:
                    weights = active * (
                        1.0 + (VQVAE_RECOVERY_STRUCTURAL_MULTIPLIER - 1.0) * structural
                    )
                    reconstruction_loss = (token_loss * weights).sum() / weights.sum().clamp_min(1.0)
                else:
                    reconstruction_loss = plain_reconstruction_loss
                loss = (reconstruction_loss + vq_loss) / VQVAE_RECOVERY_ACCUMULATION
            recovery_scaler.scale(loss).backward()
            total_loss += loss.item() * VQVAE_RECOVERY_ACCUMULATION
            steps += 1
            if steps % VQVAE_RECOVERY_ACCUMULATION == 0:
                recovery_scaler.unscale_(recovery_optimizer)
                torch.nn.utils.clip_grad_norm_(tk.vqvae.parameters(), 0.5)
                recovery_scaler.step(recovery_optimizer)
                recovery_scaler.update()
                recovery_optimizer.zero_grad(set_to_none=True)
            if steps % 500 == 0:
                print(
                    f"recovery vqvae epoca={recovery_epoch} passo={steps} "
                    f"loss={total_loss / steps:.4f} codebook_util={utilization:.4f}"
                )

        if steps % VQVAE_RECOVERY_ACCUMULATION != 0:
            recovery_scaler.unscale_(recovery_optimizer)
            torch.nn.utils.clip_grad_norm_(tk.vqvae.parameters(), 0.5)
            recovery_scaler.step(recovery_optimizer)
            recovery_scaler.update()
            recovery_optimizer.zero_grad(set_to_none=True)
        metrics = evaluate_vqvae(recovery_test_items, max_blocks=1)
        history_row = {"epoch": recovery_epoch, "loss": total_loss / max(steps, 1), **metrics}
        recovery_history.append(history_row)
        checkpoint_path = RECOVERY_VQVAE_DIR / f"vqvae_recovery_epoch_{recovery_epoch}.pt"
        torch.save({
            "model": tk.vqvae.state_dict(),
            "epoch": recovery_epoch,
            "optimizer": recovery_optimizer.state_dict(),
            "scaler": recovery_scaler.state_dict(),
            "recovery_history": recovery_history,
            "source_checkpoint": str(recovery_vqvae_source),
        }, checkpoint_path)
        print("metricas recuperacao:", metrics)
        tk.vqvae.train()
        tk.vqvae.vq.eval()

        score = (
            metrics["syntax_valid_rate"],
            metrics["exact_block_rate"],
            metrics["token_accuracy"],
        )
        best_score = (
            best_reconstruction["syntax_valid_rate"],
            best_reconstruction["exact_block_rate"],
            best_reconstruction["token_accuracy"],
        )
        preserves_reconstruction = (
            metrics["token_accuracy"] >= source_recovery_metrics["token_accuracy"] - 0.01
            and metrics["indentation_exact_rate"] >= source_recovery_metrics["indentation_exact_rate"] - 0.02
        )
        if score > best_score and preserves_reconstruction:
            best_reconstruction = history_row
            torch.save({
                "model": tk.vqvae.state_dict(),
                "epoch": recovery_epoch,
                "metrics": metrics,
            }, RECOVERY_VQVAE_DIR / "best_recovery_vqvae.pt")

    print("melhor recuperacao:", best_reconstruction)
    del recovery_optimizer, recovery_scaler
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Recuperacao do VQ-VAE desativada.")

recovery_vqvae_best = RECOVERY_VQVAE_DIR / "best_recovery_vqvae.pt"
if recovery_vqvae_best.exists():
    tk.vqvae.load_state_dict(
        torch.load(recovery_vqvae_best, map_location="cpu", weights_only=False)["model"],
        strict=True,
    )
    CODE_VQVAE_CHECKPOINT = recovery_vqvae_best
    print("VQ-VAE recuperado selecionado:", CODE_VQVAE_CHECKPOINT)
else:
    CODE_VQVAE_CHECKPOINT = source_vqvae
    tk.vqvae.load_state_dict(
        torch.load(CODE_VQVAE_CHECKPOINT, map_location="cpu", weights_only=False)["model"],
        strict=True,
    )
    print(
        "Nenhuma recuperacao superou o baseline. Mantendo o checkpoint original:",
        CODE_VQVAE_CHECKPOINT,
    )
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)

exemplos completos ate 48 tokens: {'train': 178, 'test': 5}
VQ-VAE de recuperacao: /workspace/results/ncm_codigo_full/vqvae/tokenizer_code_epoch_8.pt epoca: 8
metricas de partida comparaveis: {'token_accuracy': 0.9788359788359788, 'exact_block_rate': 0.4, 'syntax_valid_rate': 0.2, 'indentation_exact_rate': 1.0, 'samples': 5}
VRAM livre: 5.93 GiB
fase: cross-entropy lr: 5e-06
metricas recuperacao: {'token_accuracy': 0.9735449735449735, 'exact_block_rate': 0.4, 'syntax_valid_rate': 0.2, 'indentation_exact_rate': 0.8, 'samples': 5}
fase: cross-entropy lr: 5e-06
metricas recuperacao: {'token_accuracy': 0.9735449735449735, 'exact_block_rate': 0.4, 'syntax_valid_rate': 0.2, 'indentation_exact_rate': 0.8, 'samples': 5}
fase: cross-entropy lr: 5e-06
metricas recuperacao: {'token_accuracy': 0.9735449735449735, 'exact_block_rate': 0.4, 'syntax_valid_rate': 0.2, 'indentation_exact_rate': 0.8, 'samples': 5}
fase: cross-entropy lr: 5e-06
metricas recuperacao: {'token_accuracy': 0.9735449735449735, 

## Recuperacao incremental: adaptar o gerador existente

Este Stage 2 curto nao volta ao checkpoint XSum. Ele parte do melhor gerador anterior, recria os pares com o VQ-VAE recuperado e usa LR baixo. Se o Stage 1 recuperado for pior, nao execute esta celula.

In [ ]:
RUN_GENERATOR_RECOVERY = False
GENERATOR_RECOVERY_EPOCHS = 6
GENERATOR_RECOVERY_LR = 1e-5
GENERATOR_RECOVERY_BATCH_SIZE = 1
GENERATOR_RECOVERY_ACCUMULATION = 16
active_vqvae_meta = torch.load(CODE_VQVAE_CHECKPOINT, map_location="cpu", weights_only=False)
active_vqvae_epoch = int(active_vqvae_meta.get("epoch", 0))
RECOVERY_GENERATOR_DIR = (
    Path("/workspace/results/ncm_codigo_full/recovery_generator")
    / f"vqvae_epoch_{active_vqvae_epoch}"
)
RECOVERY_GENERATOR_DIR.mkdir(parents=True, exist_ok=True)

if not recovery_vqvae_best.exists():
    print(
        "Stage 2 de recuperacao bloqueado: nenhuma epoca de recuperacao superou o baseline VQ-VAE."
    )
elif RUN_GENERATOR_RECOVERY:
    if 'net' in globals():
        del net
    gc.collect()
    torch.cuda.empty_cache()
    require_free_vram(2.5)
    net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
    recovery_generator_checkpoints = sorted(
        RECOVERY_GENERATOR_DIR.glob("generator_recovery_epoch_*.pt"),
        key=lambda path: int(path.stem.split("_")[-1]),
    )
    if recovery_generator_checkpoints:
        generator_recovery_source = recovery_generator_checkpoints[-1]
        old_best = torch.load(generator_recovery_source, map_location="cpu", weights_only=False)
    else:
        generator_recovery_source = NEW_BEST_CHECKPOINT
        old_best = torch.load(generator_recovery_source, map_location="cpu", weights_only=False)
    net.load_state_dict(old_best["model"], strict=True)
    start_recovery_epoch = int(old_best.get("epoch", 0)) if recovery_generator_checkpoints else 0
    print("gerador base:", generator_recovery_source, "epoca recovery:", start_recovery_epoch)
    net.train()
    tk.vqvae.eval()
    recovery_generator_items = expand_generator_examples(train_items)
    recovery_validation_items = expand_generator_examples(validation_items)
    optimizer = torch.optim.AdamW(
        (p for p in net.parameters() if p.requires_grad),
        lr=GENERATOR_RECOVERY_LR,
        weight_decay=0.01,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    if old_best.get("optimizer") is not None:
        optimizer.load_state_dict(old_best["optimizer"])
    if old_best.get("scaler") is not None:
        scaler.load_state_dict(old_best["scaler"])
    best_recovery_generator = RECOVERY_GENERATOR_DIR / "best_recovery_generator.pt"
    if best_recovery_generator.exists():
        best_validation_loss = float(torch.load(
            best_recovery_generator, map_location="cpu", weights_only=False
        ).get("validation_loss", float("inf")))
    else:
        best_validation_loss = float("inf")
    history = list(old_best.get("history", []))
    for epoch in range(
        start_recovery_epoch + 1,
        start_recovery_epoch + GENERATOR_RECOVERY_EPOCHS + 1,
    ):
        random.shuffle(recovery_generator_items)
        optimizer.zero_grad(set_to_none=True)
        train_loss = 0.0
        steps = 0
        for start in range(0, len(recovery_generator_items), GENERATOR_RECOVERY_BATCH_SIZE):
            batch_items = recovery_generator_items[start:start + GENERATOR_RECOVERY_BATCH_SIZE]
            model_input, targets, prefix_pad_mask = make_training_batch(batch_items)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                ) / GENERATOR_RECOVERY_ACCUMULATION
            scaler.scale(loss).backward()
            train_loss += loss.item() * GENERATOR_RECOVERY_ACCUMULATION
            steps += 1
            if steps % GENERATOR_RECOVERY_ACCUMULATION == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            if steps % 500 == 0:
                print(f"recovery gerador epoca={epoch} passo={steps} loss={train_loss / steps:.4f}")
        if steps % GENERATOR_RECOVERY_ACCUMULATION != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        net.eval()
        validation_loss = 0.0
        validation_steps = 0
        with torch.inference_mode():
            for start in range(0, len(recovery_validation_items), GENERATOR_RECOVERY_BATCH_SIZE):
                batch_items = recovery_validation_items[start:start + GENERATOR_RECOVERY_BATCH_SIZE]
                model_input, targets, prefix_pad_mask = make_training_batch(batch_items)
                logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                validation_loss += F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                ).item()
                validation_steps += 1
        mean_train = train_loss / max(steps, 1)
        mean_validation = validation_loss / max(validation_steps, 1)
        history.append((epoch, mean_train, mean_validation))
        checkpoint_path = RECOVERY_GENERATOR_DIR / f"generator_recovery_epoch_{epoch}.pt"
        torch.save({
            "model": net.state_dict(),
            "epoch": epoch,
            "train_loss": mean_train,
            "validation_loss": mean_validation,
            "vqvae_checkpoint": str(CODE_VQVAE_CHECKPOINT),
            "history": history,
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
        }, checkpoint_path)
        print(
            f"recovery gerador epoca={epoch} train={mean_train:.4f} "
            f"validation={mean_validation:.4f}"
        )
        if mean_validation < best_validation_loss:
            best_validation_loss = mean_validation
            torch.save({
                "model": net.state_dict(),
                "epoch": epoch,
                "validation_loss": mean_validation,
                "vqvae_checkpoint": str(CODE_VQVAE_CHECKPOINT),
            }, RECOVERY_GENERATOR_DIR / "best_recovery_generator.pt")
        net.train()
        tk.vqvae.eval()
else:
    print("Recuperacao do gerador desativada.")

## Geracao e validacao sintetica

A geração carrega o par Stage 1 + Stage 2 correspondente. Com o VQ-VAE de codigo ativo, o checkpoint legado nunca e usado.

In [31]:
NEW_BEST_CHECKPOINT = Path("/workspace/results/ncm_codigo_full/generator/best.pt")
LEGACY_BEST_CHECKPOINT = Path("/workspace/results/ncm_codigo/best.pt")
RECOVERY_BEST_CHECKPOINT = (
    RECOVERY_GENERATOR_DIR / "best_recovery_generator.pt"
)
if recovery_vqvae_best.exists() and not RECOVERY_BEST_CHECKPOINT.exists():
    print("Recovery VQ-VAE disponivel, mas o gerador correspondente ainda nao foi treinado.")
if RECOVERY_BEST_CHECKPOINT.exists():
    NEW_BEST_CHECKPOINT = RECOVERY_BEST_CHECKPOINT
if CODE_VQVAE_CHECKPOINT is not None and not NEW_BEST_CHECKPOINT.exists():
    raise FileNotFoundError("O Stage 1 foi carregado, execute o Stage 2 antes da inferencia.")
BEST_CHECKPOINT = NEW_BEST_CHECKPOINT if CODE_VQVAE_CHECKPOINT is not None else LEGACY_BEST_CHECKPOINT
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint fine-tunado nao encontrado: {BEST_CHECKPOINT}")
best_saved = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
net.load_state_dict(best_saved["model"], strict=True)
saved_vqvae = best_saved.get("vqvae_checkpoint")
if saved_vqvae and CODE_VQVAE_CHECKPOINT is not None and Path(saved_vqvae).resolve() != Path(CODE_VQVAE_CHECKPOINT).resolve():
    raise RuntimeError(
        f"Incompatibilidade: gerador salvo com {saved_vqvae}, "
        f"mas o VQ-VAE ativo e {CODE_VQVAE_CHECKPOINT}."
    )
print(
    "checkpoint de inferencia:", BEST_CHECKPOINT,
    "epoca:", best_saved.get("epoch"),
    "validation_loss:", round(best_saved.get("validation_loss", 0.0), 4),
)

if test_items:
    sample = test_items[0]
else:
    sample = train_items[0]

prefix_ids = tk.pre_tokenizer.encode(sample["prompt"])
prefix_ids = prefix_ids[:512]
prefix_ids = prefix_ids + [PAD_ID] * (512 - len(prefix_ids))
prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)

net.eval()
with torch.inference_mode():
    concept_ids = net.generate(
        idx=None,
        prefix_tokens=prefix_tensor,
        temperature=0.2,
        top_k=1,
        top_p=0.5,
    )
    logits = tk.vqvae.decode(concept_ids)
    raw_ids = logits.squeeze(0).argmax(dim=-1)
    generated = tk.pre_tokenizer.decode(raw_ids.tolist(), skip_special_tokens=True)

print("prompt:", sample["prompt"])
print("codigo gerado:\n", generated)
try:
    ast.parse(generated)
    print("Sintaxe Python: valida")
except SyntaxError as exc:
    print("Sintaxe Python: invalida -", exc)

checkpoint de inferencia: /workspace/results/ncm_codigo_full/generator/best.pt epoca: 19 validation_loss: 1.7575
prompt: Function to handle difficult examples
            Update on each object
codigo gerado:
 def path(self, filenameNone):
       """
        """Function to handle fewer examples

         Parameters
       
Sintaxe Python: invalida - invalid syntax (<unknown>, line 3)


## Inferencia interativa

Edite `INFERENCE_PROMPT` e execute a celula para ver os codigos NCM gerados em cada nivel, os tokens GPT-2 obtidos pelo decodificador VQ-VAE e o codigo Python final.

In [32]:
from IPython.display import Code, Markdown, display

INFERENCE_PROMPT = "Write a Python function that returns the square of an integer."
INFERENCE_TEMPERATURE = 0.45
INFERENCE_TOP_K = 40
INFERENCE_TOP_P = 0.9
INFERENCE_BLOCKS = 1
INFERENCE_CANDIDATES = 8
INFERENCE_REPETITION_PENALTY = 1.25
INFERENCE_MAX_REPEATED_TOKENS = 4
INFERENCE_CHECKPOINT_EPOCHS = (5, 10, 15, 19)
INFERENCE_CHECKPOINT_EPOCH = None

def generate_concepts_without_repetition(prefix_tensor, temperature, top_k, top_p, repetition_penalty):
    original_sampler = net._sample_level_raw
    generated_codes = {}

    def sampler(logits, sample_temperature, sample_top_k, sample_top_p):
        adjusted = logits.clone()
        if repetition_penalty > 1.0 and generated_codes:
            for code, count in generated_codes.items():
                penalty = repetition_penalty ** min(count, 8)
                adjusted[..., code] /= penalty
        result = original_sampler(
            adjusted, sample_temperature, sample_top_k, sample_top_p
        )
        for code in result.reshape(-1).tolist():
            generated_codes[code] = generated_codes.get(code, 0) + 1
        return result

    net._sample_level_raw = sampler
    try:
        with torch.inference_mode():
            return net.generate(
                idx=None,
                prefix_tokens=prefix_tensor,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )
    finally:
        net._sample_level_raw = original_sampler

def trim_degenerated_tokens(token_ids, max_repeated=INFERENCE_MAX_REPEATED_TOKENS):
    eos = tk.pre_tokenizer.eos_token_id
    if eos in token_ids:
        token_ids = token_ids[:token_ids.index(eos)]
    if len(token_ids) < 2:
        return token_ids
    last = token_ids[0]
    run = 1
    for index, token in enumerate(token_ids[1:], start=1):
        run = run + 1 if token == last else 1
        if run > max_repeated:
            return token_ids[:index]
        last = token
    return token_ids

def generate_and_show(prompt, temperature=0.3, top_k=20, top_p=0.9, blocks=3):
    net.eval()
    generated_parts = []
    all_concepts = []
    all_token_ids = []
    for block_index in range(blocks):
        generated_so_far = "".join(generated_parts)
        condition = prompt
        if generated_so_far:
            condition += "\n\n# Continue the Python implementation:\n" + generated_so_far
        prompt_ids = tk.pre_tokenizer.encode(condition)
        task_ids = tk.pre_tokenizer.encode(prompt)
        if len(prompt_ids) > net.prefix_len:
            available = max(net.prefix_len - min(len(task_ids), net.prefix_len), 0)
            prompt_ids = task_ids[:net.prefix_len - available] + prompt_ids[-available:] if available else task_ids[:net.prefix_len]
        prompt_ids += [PAD_ID] * (net.prefix_len - len(prompt_ids))
        prefix_tensor = torch.tensor(
            prompt_ids, dtype=torch.long, device=device
        ).unsqueeze(0)
        with torch.inference_mode():
            concept_ids = generate_concepts_without_repetition(
                prefix_tensor,
                temperature,
                top_k,
                top_p,
                INFERENCE_REPETITION_PENALTY,
            )
            token_logits = tk.vqvae.decode(concept_ids)
            raw_ids = token_logits.squeeze(0).argmax(dim=-1)
        raw_ids = torch.tensor(
            trim_degenerated_tokens(raw_ids.tolist()),
            dtype=torch.long,
        )
        generated_part = tk.pre_tokenizer.decode(
            raw_ids.tolist(), skip_special_tokens=True
        )
        generated_parts.append(generated_part)
        all_concepts.append(concept_ids.detach().cpu())
        all_token_ids.append(raw_ids.detach().cpu())
        try:
            ast.parse("".join(generated_parts))
            break
        except SyntaxError:
            if len(set(raw_ids.tolist())) <= INFERENCE_MAX_REPEATED_TOKENS:
                break
    generated = "".join(generated_parts)

    display(Markdown("### Prompt"), Markdown(f"{prompt}"))
    display(Markdown("### Codigos NCM gerados por nivel"))
    for block_index, concept_ids in enumerate(all_concepts):
        offset = 0
        print(f"bloco {block_index + 1}")
        for level, width in enumerate(net.levels):
            values = concept_ids[0, offset:offset + width].tolist()
            print(f"  nivel {level}: {values}")
            offset += width

    display(Markdown("### Tokens GPT-2 decodificados"))
    for block_index, token_ids in enumerate(all_token_ids):
        print(f"bloco {block_index + 1}: {token_ids.tolist()}")
    display(Markdown("### Codigo gerado"))
    display(Code(generated, language="python"))

    try:
        ast.parse(generated)
        display(Markdown("**Sintaxe Python: valida**"))
    except SyntaxError as exc:
        display(Markdown(f"**Sintaxe Python: invalida** — `{exc}`"))
    return generated, all_concepts, all_token_ids

valid_candidate = None
inference_candidates = []
generator_dir = Path("/workspace/results/ncm_codigo_full/generator")
if INFERENCE_CHECKPOINT_EPOCH is not None:
    selected_checkpoint = generator_dir / f"epoch_{INFERENCE_CHECKPOINT_EPOCH}.pt"
    if not selected_checkpoint.exists():
        raise FileNotFoundError(f"Checkpoint solicitado nao existe: {selected_checkpoint}")
    selected_saved = torch.load(selected_checkpoint, map_location="cpu", weights_only=False)
    net.load_state_dict(selected_saved["model"], strict=True)
    print(
        "checkpoint selecionado para inferencia:", selected_checkpoint,
        "validation_loss:", round(selected_saved.get("validation_loss", 0.0), 4),
    )

for candidate_index in range(INFERENCE_CANDIDATES):
    display(Markdown(f"## Candidato {candidate_index + 1}"))
    candidate = generate_and_show(
        INFERENCE_PROMPT,
        temperature=INFERENCE_TEMPERATURE,
        top_k=INFERENCE_TOP_K,
        top_p=INFERENCE_TOP_P,
        blocks=INFERENCE_BLOCKS,
    )
    inference_candidates.append(candidate)
    try:
        parsed_ast = ast.parse(candidate[0])
        valid_candidate = candidate
        print("AST aceito:", type(parsed_ast.body[0]).__name__ if parsed_ast.body else "vazio")
        break
    except SyntaxError:
        continue

if valid_candidate is None:
    print("Nenhum candidato produziu uma AST valida.")

checkpoint selecionado para inferencia: /workspace/results/ncm_codigo_full/generator/epoch_19.pt validation_loss: 1.7575


## Candidato 1

### Prompt

Write a Python function that returns the square of an integer.

### Codigos NCM gerados por nivel

bloco 1
  nivel 0: [1043, 3669, 797, 797]
  nivel 1: [3690, 2551, 2435, 564, 564, 564]
  nivel 2: [487, 829, 829, 3799, 3807, 3456, 685, 2668, 685, 2668]
  nivel 3: [1330, 3299, 30, 533, 1334, 533, 1734, 679, 3129, 3129, 3129, 3129, 3129, 679]
  nivel 4: [2429, 2822, 266, 3601, 2147, 266, 2672, 2672, 266, 3760, 1945, 702, 3740, 1945, 1945, 3740, 1945, 1945, 3740, 3740]
  nivel 5: [538, 538, 1302, 2863, 2863, 85, 2465, 3937, 3215, 3117, 310, 2465, 668, 2202, 1179, 2202, 1179, 1179, 1179, 2202, 1179, 1179, 1179, 2202, 4003, 1179]
  nivel 6: [2737, 2226, 3659, 3659, 3632, 3632, 1831, 1125, 1040, 1339, 350, 350, 3161, 3082, 1125, 2758, 3419, 3419, 3419, 3419, 1155, 1155, 3419, 3419, 2758, 2758, 1155, 1155, 3419, 3419, 1155, 3419]
  nivel 7: [3356, 3356, 967, 1698, 3374, 2406, 411, 2228, 1604, 3815, 411, 2032, 547, 2131, 3767, 2445, 3210, 411, 2086, 3357, 3357, 3357, 3137, 3137, 3357, 3357, 3357, 3357, 3137, 3357, 3137, 3137, 3357, 3357, 3357, 3357, 3137, 3137, 3357, 3137]
  nivel 8: [4031,

... (72 outputs left out)

### Codigo gerado

def empty_wise(nn):
  return (i) *(n+n+i+t* the the the the

**Sintaxe Python: invalida** — `invalid syntax. Perhaps you forgot a comma? (<unknown>, line 2)`

Nenhum candidato produziu uma AST valida.


## Limitacoes

- o checkpoint XSum e apenas a inicializacao; avalie se o Stage 1 melhorou a reconstrucao antes do Stage 2;
- o VQ-VAE gera blocos de 48 tokens por chamada; sequencias maiores dependem de continuacao condicional entre blocos;
- o modelo pode produzir uma AST valida que ainda contenha erros semanticos ou falhas nos testes;
- candidates validos precisam ser executados em testes antes de uso;
- nao use o checkpoint como modelo de producao sem validacao humana e testes executaveis.